# Aufgabe 1: Betrachtung der Daten

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

df_labeled = pd.read_csv("data/buchungen.csv")
df_unlabeled = pd.read_csv("data/buchungen_test.csv")  # Datensatz ohne Storniert-Status

display(df_labeled.dtypes)  # Ausgabe der Spalten und deren jeweiligen Datentyp
display(df_labeled)

In [ ]:
df = df_labeled.copy()

cancel_rate = float(df.loc[df["storniert"] == 1]["storniert"].count() / df["storniert"].count() * 100)

df_type = df[df["hotel_typ"] == "Ferienhotel"]
cancel_rate_holiday = float(df_type[df["storniert"] == 1]["storniert"].count() / df["storniert"].count() * 100)

df_type = df[df["hotel_typ"] == "Stadthotel"]
cancel_rate_city = float(df_type[df["storniert"] == 1]["storniert"].count() / df["storniert"].count() * 100)

print(
    f"Stornierungsraten\n"
    f"Gesamt:       {cancel_rate:.2f} %\n"
    f"Ferienhotel:   {cancel_rate_holiday:.2f} %\n"
    f"Stadthotel:   {cancel_rate_city:.2f} %\n"
)

for column in df_labeled.select_dtypes(include=np.number).columns.drop(["storniert", "stammgast", "agentur_id"]):
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    fig.suptitle(f"Verteilung von {column}")

    axes[0].hist(df[column].dropna(), bins="auto", color="orange", edgecolor="black")
    axes[0].set_ylabel("Häufigkeit")

    axes[1].boxplot(df[column].dropna(), vert=False)

    plt.tight_layout()
    plt.show()

In [ ]:
from sklearn.preprocessing import OneHotEncoder


def show_correlations(df: pd.DataFrame, target: pd.DataFrame) -> None:
    df_corr = df.copy()
    df_corr["storniert"] = target.copy()

    correlations = df_corr.corr()
    display(correlations["storniert"].sort_values(ascending=False))


# Korrelation der numerischen Features / Storniert
drop_cols = ["storniert", "stammgast", "agentur_id"]

show_correlations(
    df_labeled[df_labeled.select_dtypes(include=np.number).columns.drop(drop_cols)], df_labeled["storniert"]
)

# Korrelation der kategorischen Features / Storniert
encoding = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

drop_cols = ["anreisedatum", "reservierungsstatus", "reservierungsstatus_datum"]
df_encoded = encoding.fit_transform(df_labeled[df_labeled.select_dtypes(exclude=np.number).columns.drop(drop_cols)])
df_encoded = pd.DataFrame(df_encoded, columns=encoding.get_feature_names_out(), index=df_labeled.index)

show_correlations(df_encoded, df_labeled["storniert"])